In [2]:
import subprocess, torch

# GPU 检查
print("=== GPU ===")
print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total VRAM: {total:.1f} GB")

# 包检查
print("\n=== Packages ===")
pkgs = ["transformers", "peft", "bitsandbytes", "datasets", "accelerate", "trl"]
for pkg in pkgs:
    try:
        mod = __import__(pkg)
        print(f"  {pkg}: {mod.__version__}")
    except ImportError:
        print(f"  {pkg}: NOT INSTALLED ← 需要安装")

=== GPU ===
torch version: 2.11.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3090
Total VRAM: 25.4 GB

=== Packages ===
  transformers: 5.5.4
  peft: 0.19.1
  bitsandbytes: 0.49.2
  datasets: 4.8.4
  accelerate: 1.13.0
  trl: 1.2.0


In [6]:
from huggingface_hub import list_datasets

results = list(list_datasets(search="financial sentiment", limit=10))
for d in results:
    print(d.id)

zeroshot/twitter-financial-news-sentiment
Jean-Baptiste/financial_news_sentiment
Jean-Baptiste/financial_news_sentiment_mixte_with_phrasebank_75
chiapudding/kaggle-financial-sentiment
ydqe2/kaggle_financial_sentiment_resplit
hw2942/financial-news-sentiment
intanm/indonesian-financial-sentiment-analysis
ghbacct/twitter-financial-news-sentiment-clustering
ghbacct/twitter-financial-news-sentiment-classification
jppgks/twitter-financial-news-sentiment


In [7]:
from datasets import load_dataset

dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")
print(dataset)
print("\n--- 样本示例 ---")
for i in range(3):
    print(dataset['train'][i])

README.md: 0.00B [00:00, ?B/s]

sent_train.csv: 0.00B [00:00, ?B/s]

sent_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})

--- 样本示例 ---
{'text': '$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT', 'label': 0}
{'text': '$CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3', 'label': 0}
{'text': '$CX - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook https://t.co/KN1g4AWFIb', 'label': 0}


In [8]:
from collections import Counter

label_counts = Counter(dataset['train']['label'])
print("Label 分布（train）:")
for label, count in sorted(label_counts.items()):
    print(f"  label {label}: {count} 条 ({count/len(dataset['train'])*100:.1f}%)")

print(f"\n总训练集: {len(dataset['train'])} 条")
print(f"总验证集: {len(dataset['validation'])} 条")

Label 分布（train）:
  label 0: 1442 条 (15.1%)
  label 1: 1923 条 (20.2%)
  label 2: 6178 条 (64.7%)

总训练集: 9543 条
总验证集: 2388 条


In [9]:
print("各 label 的样本：")
for label in [0, 1, 2]:
    examples = [x for x in dataset['train'] if x['label'] == label][:2]
    print(f"\n--- label {label} ---")
    for ex in examples:
        print(f"  {ex['text']}")

各 label 的样本：

--- label 0 ---
  $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT
  $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3

--- label 1 ---
  $ALTG: Dougherty & Company starts at Buy
  $AMD - AMD's Navi shows strong adoption - BofA https://t.co/WnCksfl1gX

--- label 2 ---
  $LB - MKM Partners puts a number on Victoria's Secret https://t.co/VSzHLqLBgE
  $WING - Baird returns to Wingstop bull camp https://t.co/KfPaweOVgo


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-v0.1"

# QLoRA 核心：4bit 量化配置
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4，比 INT4 精度更高
    bnb_4bit_compute_dtype=torch.float16, # 计算时临时反量化到 fp16
    bnb_4bit_use_double_quant=True,      # scale factor 也量化，再省 ~0.4GB
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Mistral 没有 pad token，用 eos 代替

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.eos_token_id

# 检查显存占用
allocated = torch.cuda.memory_allocated() / 1e9
print(f"模型加载完成")
print(f"显存占用: {allocated:.1f} GB")
print(f"模型dtype: {model.dtype}")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-v0.1
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


模型加载完成
显存占用: 3.9 GB
模型dtype: torch.bfloat16


In [11]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # Mistral 的 attention 矩阵名称
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,828,032 || all params: 7,117,500,416 || trainable%: 0.0959


In [12]:
def preprocess(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length",
    )
    tokens["labels"] = examples["label"]
    return tokens

tokenized = dataset.map(preprocess, batched=True)
tokenized = tokenized.remove_columns(["text", "label"])
tokenized.set_format("torch")

print(tokenized)
print("\n--- 第一条样本的 keys ---")
print(tokenized['train'][0].keys())
print("input_ids shape:", tokenized['train'][0]['input_ids'].shape)

Map:   0%|          | 0/9543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2388
    })
})

--- 第一条样本的 keys ---
dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([128])


In [13]:
from transformers import TrainingArguments, Trainer
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    # 每个 label 单独算准确率
    for label in [0, 1, 2]:
        mask = labels == label
        if mask.sum() > 0:
            label_acc = (preds[mask] == labels[mask]).mean()
            print(f"  label {label} acc: {label_acc:.3f} ({mask.sum()} 条)")
    return {"accuracy": acc}

training_args = TrainingArguments(
    output_dir="./mistral-qlora-sentiment",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    bf16=True,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.327903,0.290751,0.885260
2,0.212710,0.270826,0.913317
3,0.067973,0.428652,0.914992


  label 0 acc: 0.925 (347 条)
  label 1 acc: 0.935 (475 条)
  label 2 acc: 0.861 (1566 条)
  label 0 acc: 0.856 (347 条)
  label 1 acc: 0.851 (475 条)
  label 2 acc: 0.945 (1566 条)
  label 0 acc: 0.879 (347 条)
  label 1 acc: 0.859 (475 条)
  label 2 acc: 0.940 (1566 条)


TrainOutput(global_step=1791, training_loss=0.33421251189435813, metrics={'train_runtime': 2389.4414, 'train_samples_per_second': 11.981, 'train_steps_per_second': 0.75, 'total_flos': 1.5361110460543795e+17, 'train_loss': 0.33421251189435813, 'epoch': 3.0})

In [14]:
for name, param in model.named_parameters():
    if "lora_" in name:
        print(f"{name}: {param.dtype}")
        break  # 只看第一个，其他都一样

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: torch.float32


In [15]:
model.save_pretrained("./mistral-qlora-sentiment-adapter")
print("保存完成，文件列表：")
import os
for f in os.listdir("./mistral-qlora-sentiment-adapter"):
    size = os.path.getsize(f"./mistral-qlora-sentiment-adapter/{f}")
    print(f"  {f}: {size/1024:.1f} KB")

保存完成，文件列表：
  README.md: 5.1 KB
  adapter_model.safetensors: 26664.9 KB
  adapter_config.json: 1.0 KB
